<a href="https://colab.research.google.com/github/microsoft/qlib/blob/main/examples/workflow_by_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# import sys
# sys.path.insert(0, r"D:\gitdesktop\Qtrade\qlib")

import qlib
print(qlib.__file__)

import pandas as pd
from qlib.constant import REG_CN
from qlib.utils import exists_qlib_data, init_instance_by_config
from qlib.workflow import R
from qlib.workflow.record_temp import SignalRecord, PortAnaRecord
from qlib.utils import flatten_dict
from qlib.tests.data import GetData

/Users/walterswang/Documents/GitHub/qlib/qlib/__init__.py


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
provider_uri = r"/Users/walterswang/Documents/GitHub/qlib_bin/qlib_bin_norm"  # target_dir
qlib.init(provider_uri=provider_uri, region=REG_CN)

[78367:MainThread](2025-09-13 20:31:27,178) INFO - qlib.Initialization - [config.py:452] - default_conf: client.
[78367:MainThread](2025-09-13 20:31:27,179) INFO - qlib.Initialization - [__init__.py:79] - qlib successfully initialized based on client settings.
[78367:MainThread](2025-09-13 20:31:27,179) INFO - qlib.Initialization - [__init__.py:81] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/walterswang/Documents/GitHub/qlib_bin/qlib_bin_norm')}


In [3]:
market = "all"
benchmark = "SH000300"

In [4]:
showDataDetail = False
printPredResult = False

labelNum = 16

# train model

In [5]:
###################################
# train model
###################################
data_handler_config = {
    "start_time": "2008-01-01",
    "end_time": "2024-08-31",
    "fit_start_time": "2008-01-01",
    "fit_end_time": "2020-12-31",                   # 用来这段时间控制的是 数据预处理时，用来计算“全局参数”的时间范围，比如训练数据中所有特征的均值和方差。
    "instruments": market,
    "label_days_offset": labelNum,
}

task = {
    "model": {
        "class": "LGBModel",
        "module_path": "qlib.contrib.model.gbdt",
        "kwargs": {
            "device": "cpu",
            "loss": "mse",
            "colsample_bytree": 0.8879,
            "learning_rate": 0.0421,
            "subsample": 0.8789,
            "lambda_l1": 205.6999,
            "lambda_l2": 580.9768,
            "max_depth": 8,
            "num_leaves": 210,
            "num_threads": 20,
        },
    },
    "dataset": {
        "class": "DatasetH",
        "module_path": "qlib.data.dataset",
        "kwargs": {
            "handler": {
                "class": "Alpha158",  #158 / 360
                "module_path": "qlib.contrib.data.handler",
                "kwargs": data_handler_config,
            },
            "segments": {
                "train": ("2008-01-01", "2020-08-31"),           # 训练模型的样本
                "valid": ("2020-09-01", "2021-08-31"),           # 调参或早停用的验证集
                "test": ("2021-09-01", "2024-08-31"),            # 回测阶段评估模型表现的数据集
            },
        },
    },
}

In [6]:
dataset = init_instance_by_config(task["dataset"])
if showDataDetail:
    df = dataset.handler.fetch(col_set="feature")  # 或 col_set="all" 也行
    print("数据开始日期：", df.index.min())
    print("数据结束日期：", df.index.max())
    print("数据的 features 列：", df.columns.tolist())
    # 1. 统计每只股票的最早日期
    first_dates = df.groupby(df.index.get_level_values(1)).apply(
        lambda x: x.index.get_level_values(0).min()
    )
    
    # 2. 变成DataFrame
    first_dates_df = first_dates.reset_index()
    first_dates_df.columns = ['股票代码', '入市时间']
    
    # 3. 按入市时间从晚到早排序
    first_dates_df = first_dates_df.sort_values('入市时间', ascending=False)
    
    # 4. 打印最晚入市的股票
    most_recent_stock = first_dates_df.iloc[0]
    print("最晚入市的股票:")
    print(f"股票代码：{most_recent_stock['股票代码']}")
    print(f"入市时间：{most_recent_stock['入市时间']}")
    
    # 5. 打印全部列表
    print("股票入市时间排序列表:")
    print(first_dates_df)

[78367:MainThread](2025-09-13 20:32:22,158) INFO - qlib.timer - [log.py:127] - Time cost: 54.967s | Loading data Done
[78367:MainThread](2025-09-13 20:32:25,889) INFO - qlib.timer - [log.py:127] - Time cost: 1.492s | DropnaLabel Done
[78367:MainThread](2025-09-13 20:32:28,604) INFO - qlib.timer - [log.py:127] - Time cost: 2.715s | CSZScoreNorm Done
[78367:MainThread](2025-09-13 20:32:28,770) INFO - qlib.timer - [log.py:127] - Time cost: 6.611s | fit & process data Done
[78367:MainThread](2025-09-13 20:32:28,770) INFO - qlib.timer - [log.py:127] - Time cost: 61.579s | Init data Done


In [7]:
model = init_instance_by_config(task["model"])
# start exp to train model
with R.start(experiment_name="train_model"):
    R.log_params(**flatten_dict(task))
    model.fit(dataset)
    R.save_objects(trained_model=model)
    rid = R.get_recorder().id

[78367:MainThread](2025-09-13 20:32:29,810) INFO - qlib.workflow - [exp.py:258] - Experiment 964597384787921693 starts running ...
[78367:MainThread](2025-09-13 20:32:30,032) INFO - qlib.workflow - [recorder.py:345] - Recorder 292b8aa49dcf4e44a7748db9c6802365 starts running under Experiment 964597384787921693 ...


Training until validation scores don't improve for 50 rounds
[20]	train's l2: 0.943962	valid's l2: 0.991273
[40]	train's l2: 0.928	valid's l2: 0.995698
[60]	train's l2: 0.921612	valid's l2: 1.00005


[78367:MainThread](2025-09-13 20:33:01,533) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


Early stopping, best iteration is:
[15]	train's l2: 0.951727	valid's l2: 0.990983


In [8]:
with R.start(experiment_name="backtest_analysis"):
    # 是获取训练阶段的 recorder，从中加载已经训练好的模型对象
    recorder = R.get_recorder(recorder_id=rid, experiment_name="train_model")
    model = recorder.load_object("trained_model")

    # prediction
    # 新建一个 recorder，用于记录本次预测和回测的结果
    recorder = R.get_recorder()
    ba_rid = recorder.id
    # 用训练好的模型对指定数据集进行预测，生成“信号分数”并记录到实验recorder中。
    # 这些信号分数（通常是每只股票每天的预测收益分数）会被后续的回测策略用来选股。
    sr = SignalRecord(model, dataset, recorder)
    sr.generate()

[78367:MainThread](2025-09-13 20:33:01,539) INFO - qlib.workflow - [exp.py:258] - Experiment 774061127372891962 starts running ...
[78367:MainThread](2025-09-13 20:33:01,543) INFO - qlib.workflow - [recorder.py:345] - Recorder cb30019d7034418683493f55318bce7b starts running under Experiment 774061127372891962 ...


[78367:MainThread](2025-09-13 20:33:03,560) INFO - qlib.workflow - [record_temp.py:198] - Signal record 'pred.pkl' has been saved as the artifact of the Experiment 774061127372891962
[78367:MainThread](2025-09-13 20:33:03,635) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


'The following are prediction results of the LGBModel model.'
                          score
datetime   instrument          
2021-09-01 SH000300   -0.049488
           SH000852   -0.031245
           SH000905   -0.051493
           SH000906   -0.044661
           SH000985   -0.056527


In [9]:
from qlib.contrib.report import analysis_model, analysis_position
from qlib.data import D

recorder = R.get_recorder(recorder_id=ba_rid, experiment_name="backtest_analysis")
pred_df = recorder.load_object("pred.pkl")
pred_df.to_pickle(f"score{labelNum}.bin")   # 保存

In [10]:
if printPredResult:
    test_df = pd.read_pickle(f"score{labelNum}.bin")   # 读取
    print("Prediction DataFrame.....................................................................\n")
    print("Index information:\n")
    print(test_df.index)        # 查看索引的具体内容
    print("Column information:\n")
    for col in test_df.columns:
        dtype = str(test_df[col].dtype)
        example = test_df[col].dropna().iloc[0] if not test_df[col].dropna().empty else "N/A"
        print(f"  - {col:<15} | dtype: {dtype:<10} | example: {example}")